# Final model per familie

## Imports

In [13]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import numpy as np
import sys
import warnings
from sklearn.pipeline import Pipeline

project_dir = Path.cwd().parent

sys.path.insert(0, str(Path.cwd().parent)) # import src/
warnings.filterwarnings("ignore")

pd.set_option("display.width", 220, "display.max_columns", 30)

from src import (NestedCVRegressor, default_models,
                           default_param_spaces, MRMRSelector, run_per_family,
                           save_model)

from src import best_params_from_result, build_xy, load_anndata, load_per_family, load_model

HVG_DATA_DIR = project_dir / 'data' / 'HVGs'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Dataset

In [9]:
FAMILIES = {
    "KenyonCells":  (HVG_DATA_DIR / "Kenyon_cells_train_hvg.h5ad",  HVG_DATA_DIR / "Kenyon_cells_test_hng.h5ad"),
    "OpticLobe":    (HVG_DATA_DIR / "Optic_lobe_train_hvg.h5ad",    HVG_DATA_DIR / "Optic_lobe_test_hng.h5ad"),
    "Monoaminergic":(HVG_DATA_DIR / "Monoaminergic_train_hvg.h5ad",HVG_DATA_DIR / "Monoaminergic_test_hng.h5ad"),
    "Glia":         (HVG_DATA_DIR / "Glia_train_hvg.h5ad",         HVG_DATA_DIR / "Glia_test_hng.h5ad"),
    "Peptidergic":  (HVG_DATA_DIR / "Peptidergic_train_hvg.h5ad",  HVG_DATA_DIR / "Peptidergic_test_hng.h5ad"),
    "Clock":        (HVG_DATA_DIR / "Clock_train_hvg.h5ad",        HVG_DATA_DIR / "Clock_test_hng.h5ad"),
}
FAMILIES = {f: p for f, p in FAMILIES.items() if p[0].exists() and p[1].exists()}

## Training

In [10]:
final_model_per_family = run_per_family(
    FAMILIES,
    models={"XGBoost": default_models()["XGBoost"]},
    param_spaces={"XGBoost": default_param_spaces()["XGBoost"]},
    feature_selector=MRMRSelector(k=100, relevance="f"),
    fs_mode="inner",
    tune=True,
    dev="train",
    n_rounds=10,
    n_outer=5,
    n_inner=3,
    n_trials=30,
    inner_scoring="neg_root_mean_squared_error",
    keep_models=False,
    save_dir="../results/final_model_per_family",
    save_name="final_model_per_family",
    verbose=1,
)


=== Family: KenyonCells  (2278 cells, 2001 features) ===
[KenyonCells] XGBoost: tuning ...

=== Family: OpticLobe  (6365 cells, 2001 features) ===
[OpticLobe] XGBoost: tuning ...

=== Family: Monoaminergic  (748 cells, 2001 features) ===
[Monoaminergic] XGBoost: tuning ...

=== Family: Glia  (2991 cells, 2001 features) ===
[Glia] XGBoost: tuning ...

=== Family: Peptidergic  (907 cells, 2001 features) ===
[Peptidergic] XGBoost: tuning ...

=== Family: Clock  (412 cells, 2001 features) ===
[Clock] XGBoost: tuning ...


In [11]:

for fam, (train, _test) in FAMILIES.items():
    X, y, _ = build_xy(load_anndata(train), extra_obs=("nUMI",))
    bp = best_params_from_result(final_model_per_family.results[fam], "XGBoost", metric="RMSE")
    pipe = Pipeline([("selector", MRMRSelector(k=100, relevance="f")),
                     ("model", default_models()["XGBoost"].set_params(**bp))])
    pipe.fit(X, y) # mRMR picks 100 genes on ALL train, then XGB fits
    save_model(pipe, f"../results/final_model_per_family/{fam}/final_XGBoost.joblib")

In [30]:
# CV results back from disk
final_model_per_family_loaded = load_per_family("../results/final_model_per_family",
                                         name="final_model_per_family")
cv_summary_all_families = final_model_per_family_loaded.summary
clock_features = final_model_per_family_loaded.results["Clock"].feature_stability # selected-gene frequencies
clock_params = final_model_per_family_loaded.results["Clock"].best_params # tuned params

# Predict on test sets
test_scores = {}
for fam, (train, test) in FAMILIES.items():
    Xte, yte, _ = build_xy(load_anndata(test), extra_obs=("nUMI",))
    pipe = load_model(f"../results/final_model_per_family/{fam}/final_XGBoost.joblib")
    pred = pipe.predict(Xte)
    test_scores[fam] = {"MAE": float(np.mean(np.abs(yte - pred))),
                        "RMSE": float(np.sqrt(np.mean((yte - pred) ** 2)))}
    cv_summary_per_family = final_model_per_family_loaded.results[fam].summary
print(test_scores)

{'KenyonCells': {'MAE': 2.5239391496372328, 'RMSE': 4.50197488759063}, 'OpticLobe': {'MAE': 2.6848118155586285, 'RMSE': 4.980187648027001}, 'Monoaminergic': {'MAE': 4.201943428354695, 'RMSE': 6.483696837328726}, 'Glia': {'MAE': 5.552283818676649, 'RMSE': 7.623144538554475}, 'Peptidergic': {'MAE': 4.2266997153407155, 'RMSE': 6.662715952526691}, 'Clock': {'MAE': 4.833009688756787, 'RMSE': 7.6398117828275005}}


In [27]:
best_per_type_rmse_ci = (
    cv_summary_all_families
    .loc[cv_summary_all_families.groupby("family")["RMSE_median"].idxmin()]
    .sort_values("RMSE_median")
    [["family", "model",
      "RMSE_median", "RMSE_lo", "RMSE_hi",
      "MAE_median",  "MAE_lo",  "MAE_hi",
      "R2_median",   "R2_lo",   "R2_hi",
      "Pearson_median", "Pearson_lo", "Pearson_hi",
      "Spearman_median", "Spearman_lo", "Spearman_hi"]]
)
# display(best_per_type_rmse_ci)

disp = best_per_type_rmse_ci.copy()
for m in ["RMSE", "MAE", "R2", "Pearson", "Spearman"]:
    disp[m] = disp.apply(
        lambda r: f"{r[f'{m}_median']:.3f} ({r[f'{m}_lo']:.3f}-{r[f'{m}_hi']:.3f})", axis=1)
disp = disp[["family", "model", "RMSE", "MAE", "R2", "Pearson", "Spearman"]]
display(disp)

,family,model,RMSE,MAE,R2,Pearson,Spearman
1,OpticLobe,XGBoost,4.613 (4.569-4.635),2.510 (2.470-2.564),0.812 (0.809-0.815),0.904 (0.902-0.905),0.889 (0.884-0.894)
0,KenyonCells,XGBoost,5.088 (5.017-5.170),2.786 (2.755-2.815),0.732 (0.722-0.741),0.860 (0.853-0.867),0.847 (0.843-0.856)
2,Monoaminergic,XGBoost,7.027 (6.864-7.193),4.330 (4.259-4.453),0.692 (0.682-0.710),0.843 (0.835-0.852),0.806 (0.790-0.821)
4,Peptidergic,XGBoost,7.027 (6.887-7.198),4.385 (4.330-4.468),0.685 (0.670-0.697),0.834 (0.824-0.841),0.809 (0.802-0.819)
3,Glia,XGBoost,7.999 (7.930-8.113),5.567 (5.537-5.645),0.828 (0.823-0.831),0.910 (0.908-0.913),0.909 (0.905-0.911)
5,Clock,XGBoost,8.672 (8.349-8.844),5.377 (5.166-5.627),0.563 (0.546-0.589),0.770 (0.757-0.785),0.786 (0.764-0.803)


In [34]:
display(clock_features.head(10))

,gene,frequency
0,nUMI,1.0
1,CG9377,1.0
2,Fas1,1.0
3,RpS14b,1.0
4,Gapdh1,1.0
5,Ggamma1,1.0
6,Pur-alpha,1.0
7,nrv2,1.0
8,CG13117,1.0
9,mt:lrRNA,1.0


In [35]:
display(clock_params)

{('XGBoost', 0, 0): {'n_estimators': 348,
  'max_depth': 3,
  'learning_rate': 0.039715189090884596,
  'subsample': 0.7488254993831445,
  'colsample_bytree': 0.8681909152054504,
  'reg_alpha': 3.69064504430829,
  'reg_lambda': 3.615977153631591},
 ('XGBoost', 0, 1): {'n_estimators': 268,
  'max_depth': 5,
  'learning_rate': 0.01759062458921286,
  'subsample': 0.9553852887955392,
  'colsample_bytree': 0.76049607137546,
  'reg_alpha': 0.031248713202851566,
  'reg_lambda': 0.021991975211503017},
 ('XGBoost', 0, 2): {'n_estimators': 207,
  'max_depth': 8,
  'learning_rate': 0.03900109488032909,
  'subsample': 0.8402199441428595,
  'colsample_bytree': 0.7905758989750167,
  'reg_alpha': 1.2624777200098898,
  'reg_lambda': 0.006208781903062114},
 ('XGBoost', 0, 3): {'n_estimators': 416,
  'max_depth': 3,
  'learning_rate': 0.11484099019415706,
  'subsample': 0.7535856699089988,
  'colsample_bytree': 0.9748897406719734,
  'reg_alpha': 0.04240982703632249,
  'reg_lambda': 1.7601152392931645},
 